# 📖 Module 05: Vector Databases

## GenAI L2 Exam Preparation

**Topics Covered:**
- What are vector databases and why we need them
- FAISS — Fast local search
- ChromaDB — Easy local development
- Pinecone — Cloud-managed solution
- Qdrant — Cloud or local with advanced filtering
- Comparison matrix and selection guide

**Source Material:** Class 33 (FAISS & Vector DB), Class 34 (ChromaDB, Pinecone, Qdrant)

---

## 1. What is a Vector Database?

A vector database is a specialized database designed to **store, index, and search** high-dimensional vectors (embeddings).

### Why Not Use a Regular Database?

| Feature | Regular DB (SQL) | Vector DB |
|---------|-----------------|----------|
| **Search type** | Exact match, keyword | Similarity/nearest neighbor |
| **Data type** | Structured (rows/columns) | Vectors (high-dimensional) |
| **Query** | `WHERE name = 'RAG'` | "Find vectors closest to this query vector" |
| **Speed for similarity** | Very slow | Optimized (ANN algorithms) |
| **Use case** | CRUD operations | Semantic search, recommendations |

### How Vector DBs Work

```
1. INDEX: Store vectors with metadata
   vector=[0.1, 0.3, ...], metadata={"source": "doc1.pdf", "page": 3}

2. SEARCH: Find nearest neighbors
   query_vector → ANN algorithm → Top-K most similar vectors

3. RETURN: Documents + scores
   [{doc: "...", score: 0.95}, {doc: "...", score: 0.87}, ...]
```

### ANN (Approximate Nearest Neighbor)
- Exact search is O(n) — too slow for millions of vectors
- ANN algorithms (HNSW, IVF, PQ) trade **small accuracy loss** for **massive speed gains**
- Most vector DBs use HNSW (Hierarchical Navigable Small World) graphs

### 🎯 Exam Tip
> Vector DBs use **ANN algorithms** (not exact search) for speed.  
> This means results are **approximate** but very fast.

In [ ]:
# Setup
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Common embeddings for all examples
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Sample documents for all examples
sample_docs = [
    Document(page_content="RAG combines retrieval systems with generative AI for accurate responses.", metadata={"source": "rag_guide.pdf", "topic": "RAG"}),
    Document(page_content="Vector databases store embeddings and enable fast similarity search.", metadata={"source": "vectordb_guide.pdf", "topic": "VectorDB"}),
    Document(page_content="FAISS is a library for efficient similarity search developed by Facebook AI.", metadata={"source": "faiss_docs.pdf", "topic": "VectorDB"}),
    Document(page_content="ChromaDB is an open-source embedding database perfect for local development.", metadata={"source": "chroma_docs.pdf", "topic": "VectorDB"}),
    Document(page_content="Chunking splits documents into smaller pieces for better retrieval precision.", metadata={"source": "rag_guide.pdf", "topic": "Chunking"}),
    Document(page_content="Embeddings convert text into numerical vectors capturing semantic meaning.", metadata={"source": "embedding_guide.pdf", "topic": "Embeddings"}),
    Document(page_content="Pinecone is a fully managed vector database service for production use.", metadata={"source": "pinecone_docs.pdf", "topic": "VectorDB"}),
    Document(page_content="Fine-tuning modifies model weights while RAG retrieves external knowledge.", metadata={"source": "rag_guide.pdf", "topic": "RAG"}),
]

print(f"✅ Loaded {len(sample_docs)} sample documents")
print(f"📐 Embedding dimension: {len(embeddings.embed_query('test'))}")

## 2. FAISS (Facebook AI Similarity Search)

| Feature | Detail |
|---------|--------|
| **Type** | Library (not a database) |
| **Deployment** | Local only |
| **Speed** | ⭐⭐⭐⭐⭐ (fastest) |
| **Scalability** | Millions of vectors locally |
| **Persistence** | Manual save/load |
| **Filtering** | Limited |
| **Best For** | Fast local prototyping, read-heavy workloads |

In [ ]:
from langchain_community.vectorstores import FAISS

# Create FAISS vector store
faiss_db = FAISS.from_documents(sample_docs, embeddings)
print("✅ FAISS vector store created")

# Similarity search
query = "What is a vector database?"
results = faiss_db.similarity_search(query, k=3)

print(f"\n🔍 Query: '{query}'")
print(f"📄 Top {len(results)} results:")
for i, doc in enumerate(results, 1):
    print(f"  [{i}] {doc.page_content}")
    print(f"      Metadata: {doc.metadata}")

In [ ]:
# Similarity search WITH SCORES
results_with_scores = faiss_db.similarity_search_with_score(query, k=3)

print(f"🔍 Results with scores (lower = more similar in FAISS):")
for doc, score in results_with_scores:
    print(f"  Score: {score:.4f} | {doc.page_content[:60]}...")

In [ ]:
# FAISS: Save and Load
import os

save_path = "./data/faiss_index"

# Save
faiss_db.save_local(save_path)
print(f"💾 Saved FAISS index to {save_path}")

# Load (note: allow_dangerous_deserialization=True is required)
loaded_db = FAISS.load_local(
    save_path, 
    embeddings, 
    allow_dangerous_deserialization=True  # Only use with trusted data!
)
print(f"📂 Loaded FAISS index from {save_path}")

# Verify it works
results = loaded_db.similarity_search("vector search", k=1)
print(f"✅ Verification: {results[0].page_content[:60]}...")

### 🎯 FAISS Exam Tips
> - FAISS uses **L2 distance** (Euclidean) by default — lower score = more similar
> - Must call `save_local()` explicitly — FAISS does NOT auto-persist
> - `allow_dangerous_deserialization=True` is needed for `load_local()` — only use with trusted data
> - Best for **speed-critical** applications with **local data**

## 3. ChromaDB (Recommended for Development)

| Feature | Detail |
|---------|--------|
| **Type** | Embedded database |
| **Deployment** | Local |
| **Speed** | ⭐⭐⭐ (good) |
| **Ease of use** | ⭐⭐⭐⭐⭐ (easiest) |
| **Persistence** | Auto-persist to disk |
| **Filtering** | Good metadata filtering |
| **Best For** | Local development, prototyping, learning |

In [ ]:
try:
    from langchain_chroma import Chroma
    
    # Create ChromaDB (auto-persists to disk)
    chroma_db = Chroma.from_documents(
        documents=sample_docs,
        embedding=embeddings,
        persist_directory="./data/chroma_db",
        collection_name="exam_prep"
    )
    print("✅ ChromaDB vector store created")
    
    # Search
    results = chroma_db.similarity_search("What is RAG?", k=3)
    print(f"\n🔍 Top 3 results for 'What is RAG?':")
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] {doc.page_content}")

except ImportError:
    print("⚠️ langchain_chroma not installed. Run: pip install langchain_chroma")

In [ ]:
# ChromaDB: Load existing collection
try:
    existing_db = Chroma(
        persist_directory="./data/chroma_db",
        embedding_function=embeddings,
        collection_name="exam_prep"
    )
    
    # Search with scores
    results = existing_db.similarity_search_with_score("FAISS", k=2)
    print("📊 Results with scores (higher = more similar in Chroma):")
    for doc, score in results:
        print(f"  Score: {score:.4f} | {doc.page_content[:60]}...")
        
except Exception as e:
    print(f"⚠️ {e}")

### 🎯 ChromaDB Exam Tips
> - ChromaDB **auto-persists** — no need for manual save
> - Uses **cosine similarity** by default — higher score = more similar
> - Supports **metadata filtering**: `where={"topic": "RAG"}`
> - Easiest to set up — great for development and learning

## 4. Pinecone (Cloud Production)

| Feature | Detail |
|---------|--------|
| **Type** | Fully managed cloud service |
| **Deployment** | Cloud only |
| **Speed** | ⭐⭐⭐⭐ (fast, distributed) |
| **Scalability** | ⭐⭐⭐⭐⭐ (billions of vectors) |
| **Filtering** | Advanced metadata filtering |
| **Best For** | Production, large-scale applications |

```python
# Pinecone Setup (requires API key)
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
import os

# Initialize Pinecone client
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Create index (one-time operation)
pc.create_index(
    name="my-rag-index",
    dimension=384,              # Must match embedding model dimension!
    metric="cosine",            # cosine, euclidean, or dotproduct
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

# Create vector store from documents
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name="my-rag-index"
)

# Search
results = vectorstore.similarity_search("What is RAG?", k=5)
```

### 🎯 Pinecone Exam Tips
> - **dimension** in `create_index()` MUST match embedding model dimension
> - Supports **serverless** and **pod-based** deployment
> - Free tier has limits (100K vectors, 1 index)
> - Best for **production** workloads needing managed infrastructure

## 5. Qdrant (Versatile — Cloud or Local)

| Feature | Detail |
|---------|--------|
| **Type** | Vector search engine |
| **Deployment** | Local, Docker, or Cloud |
| **Speed** | ⭐⭐⭐⭐ (fast) |
| **Filtering** | ⭐⭐⭐⭐⭐ (best filtering) |
| **Scalability** | ⭐⭐⭐⭐⭐ (excellent) |
| **Best For** | When you need advanced filtering, flexible deployment |

```python
# Qdrant Setup
from langchain_qdrant import QdrantVectorStore

# In-memory (for testing)
vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    location=":memory:",
    collection_name="my_collection"
)

# Local persistent
vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    path="./qdrant_data",
    collection_name="my_collection"
)

# Cloud
vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    url="https://your-cluster.qdrant.io",
    api_key=os.getenv("QDRANT_API_KEY"),
    collection_name="my_collection"
)
```

### 🎯 Qdrant Exam Tips
> - Most **versatile** — works in-memory, local disk, Docker, or cloud
> - **Best filtering capabilities** of all vector DBs
> - Supports **payload** (rich metadata) with complex filter expressions

## 6. ⭐ Vector Database Comparison Matrix (EXAM CRITICAL!)

| Feature | FAISS | ChromaDB | Pinecone | Qdrant |
|---------|-------|----------|----------|--------|
| **Type** | Library | Database | Managed Service | Search Engine |
| **Deployment** | Local only | Local only | Cloud only | Local + Cloud |
| **Setup ease** | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Speed** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Scalability** | ⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Filtering** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Persistence** | Manual | Auto | Auto (cloud) | Auto |
| **Free tier** | ✅ (fully free) | ✅ (fully free) | ✅ (limited) | ✅ (limited) |
| **Default metric** | L2 (Euclidean) | Cosine | Cosine | Cosine |
| **Best for** | Fast prototyping | Development | Production | Flexible needs |

### Decision Guide

```
Need the fastest local search?           → FAISS
Easiest setup for learning/dev?          → ChromaDB ⭐
Production with managed infrastructure?  → Pinecone
Need advanced filtering + flexibility?   → Qdrant
Budget-conscious production?             → Qdrant (self-hosted)
```

### 🎯 Exam Tip
> Exam scenario: "Which vector DB for a production app with millions of users?" → **Pinecone** (managed) or **Qdrant Cloud**  
> Exam scenario: "Quick prototype for a hackathon?" → **ChromaDB** or **FAISS**  
> Exam scenario: "Need to filter by department, date, and category?" → **Qdrant** (best filtering)

## 7. Common Operations Summary

| Operation | FAISS | ChromaDB | Pinecone |
|-----------|-------|----------|----------|
| **Create** | `FAISS.from_documents()` | `Chroma.from_documents()` | `PineconeVectorStore.from_documents()` |
| **Search** | `.similarity_search()` | `.similarity_search()` | `.similarity_search()` |
| **Search+Score** | `.similarity_search_with_score()` | `.similarity_search_with_score()` | `.similarity_search_with_score()` |
| **Save** | `.save_local(path)` | Auto (persist_directory) | Auto (cloud) |
| **Load** | `FAISS.load_local(path, emb)` | `Chroma(persist_directory=)` | `PineconeVectorStore(index_name=)` |
| **As Retriever** | `.as_retriever()` | `.as_retriever()` | `.as_retriever()` |

## 🧠 Self-Assessment Quiz

---

**Q1.** Your company needs a vector database for a production app serving 1M users. Which would you recommend?

<details>
<summary>Click for Answer</summary>

**Pinecone** — it's a fully managed cloud service designed for production scale. It handles infrastructure, scaling, and reliability automatically. **Qdrant Cloud** is also a valid option.
</details>

---

**Q2.** What happens if you create a Pinecone index with `dimension=384` but use an embedding model that produces 768-dimensional vectors?

<details>
<summary>Click for Answer</summary>

You'll get a **dimension mismatch error**. The index dimension must exactly match the embedding model's output dimension. You'd need to either change the model or recreate the index with `dimension=768`.
</details>

---

**Q3.** Which vector DB requires `allow_dangerous_deserialization=True` when loading?

<details>
<summary>Click for Answer</summary>

**FAISS** — because it uses Python's pickle for serialization, which can execute arbitrary code. The flag is a safety reminder to only load trusted data.
</details>

---

**Q4.** Which vector database has the best metadata filtering capabilities?

<details>
<summary>Click for Answer</summary>

**Qdrant** — it supports complex filter expressions with payload (rich metadata), including range queries, nested filters, and boolean combinations.
</details>

---

**Q5.** What is the difference between FAISS's default distance metric and ChromaDB's?

<details>
<summary>Click for Answer</summary>

- **FAISS**: Uses **L2 (Euclidean) distance** by default — **lower score = more similar**  
- **ChromaDB**: Uses **cosine similarity** by default — **higher score = more similar**  
This is important when interpreting search scores!
</details>

---

## ✅ Module 5 Complete!

**Key Takeaways:**
1. Vector DBs specialize in storing/searching embeddings using ANN algorithms
2. FAISS = fastest local, ChromaDB = easiest dev, Pinecone = production cloud, Qdrant = best filtering
3. Embedding dimension MUST match between model and vector store
4. FAISS needs manual save; ChromaDB auto-persists
5. All provide `.as_retriever()` for LangChain integration

**Next:** [Module 06 — Retrievers](./06_Retrievers.ipynb)